In [ ]:
# Run this in Colab.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
from pathlib import Path
from io import BytesIO

import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Update this path to your Google Drive folder.
DATA_ROOT = Path('/content/drive/MyDrive/Fakeedit/Multimodal_samples')

# Optional image cache in Google Drive.
IMAGE_CACHE_DIR = Path('/content/drive/MyDrive/Fakeedit/image_cache')
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Image loading mode:
# 'url'   -> always load image from image_url, no local saving
# 'drive' -> use local image cache in Google Drive
IMAGE_MODE = 'url'

# If IMAGE_MODE='drive', choose whether missing images should be downloaded to Drive.
DOWNLOAD_MISSING_TO_DRIVE = True

TRAIN_TSV = DATA_ROOT / 'multimodal_train.tsv'
VAL_TSV = DATA_ROOT / 'multimodal_validate.tsv'
TEST_TSV = DATA_ROOT / 'multimodal_test_public.tsv'

print('train exists:', TRAIN_TSV.exists())
print('validate exists:', VAL_TSV.exists())
print('test exists:', TEST_TSV.exists())
print('image mode:', IMAGE_MODE)

In [ ]:
def load_fakeddit_tsv(tsv_path: str | Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(tsv_path, sep='\t')
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')].copy()

    if 'submission_id' not in df.columns and 'id' in df.columns:
        df['submission_id'] = df['id']

    required_cols = ['clean_title', 'image_url', '6_way_label', '2_way_label']
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns in {tsv_path}: {missing}')

    df['clean_title'] = df['clean_title'].fillna('').astype(str)
    df['image_url'] = df['image_url'].fillna('').astype(str)
    df['split'] = split_name

    df = df[df['clean_title'].str.strip() != ''].reset_index(drop=True)
    return df


train_df = load_fakeddit_tsv(TRAIN_TSV, 'train')
val_df = load_fakeddit_tsv(VAL_TSV, 'validate')
test_df = load_fakeddit_tsv(TEST_TSV, 'test')

print('train:', train_df.shape)
print('validate:', val_df.shape)
print('test:', test_df.shape)

display(train_df[['submission_id', 'clean_title', 'image_url', '2_way_label', '6_way_label']].head())

In [ ]:
print('6-way label distribution')
display(train_df['6_way_label'].value_counts().sort_index())

print('2-way label distribution')
display(train_df['2_way_label'].value_counts().sort_index())

In [ ]:
class FakedditColabDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        image_mode: str = 'url',
        image_cache_dir: str | Path | None = None,
        download_missing_to_drive: bool = False,
        image_size: tuple[int, int] = (224, 224),
        timeout: int = 15,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_mode = image_mode
        self.image_cache_dir = Path(image_cache_dir) if image_cache_dir is not None else None
        self.download_missing_to_drive = download_missing_to_drive
        self.image_size = image_size
        self.timeout = timeout
        self.fallback_image = Image.new('RGB', image_size, color=(0, 0, 0))

        if self.image_mode not in {'url', 'drive', 'none'}:
            raise ValueError("image_mode must be one of: 'url', 'drive', 'none'")

        if self.image_cache_dir is not None:
            self.image_cache_dir.mkdir(parents=True, exist_ok=True)

    def __len__(self):
        return len(self.df)

    def _local_path(self, sample_id: str) -> Path | None:
        if self.image_cache_dir is None:
            return None
        return self.image_cache_dir / f'{sample_id}.jpg'

    def _load_from_url(self, image_url: str) -> Image.Image:
        try:
            response = requests.get(
                image_url,
                timeout=self.timeout,
                headers={'User-Agent': 'Mozilla/5.0'},
            )
            response.raise_for_status()
            return Image.open(BytesIO(response.content)).convert('RGB')
        except Exception:
            return self.fallback_image.copy()

    def _load_drive_or_download(self, image_url: str, sample_id: str) -> Image.Image:
        local_path = self._local_path(sample_id)

        if local_path is not None and local_path.exists():
            try:
                return Image.open(local_path).convert('RGB')
            except Exception:
                pass

        if self.download_missing_to_drive and local_path is not None and image_url.strip():
            result = download_image_to_drive(image_url, sample_id, self.image_cache_dir, timeout=self.timeout)
            if result is not None:
                try:
                    return Image.open(result).convert('RGB')
                except Exception:
                    pass

        return self.fallback_image.copy()

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        sample_id = str(row['submission_id'])
        text = row['clean_title']
        image_url = row['image_url']
        label_6 = int(row['6_way_label'])
        label_2 = int(row['2_way_label'])

        if self.image_mode == 'none':
            image = None
        elif self.image_mode == 'url':
            image = self._load_from_url(image_url) if image_url.strip() else self.fallback_image.copy()
        else:
            image = self._load_drive_or_download(image_url, sample_id)

        return {
            'id': sample_id,
            'text': text,
            'image_url': image_url,
            'image': image,
            'label_6': label_6,
            'label_2': label_2,
        }


def fakeddit_collate_fn(batch):
    return {
        'id': [x['id'] for x in batch],
        'text': [x['text'] for x in batch],
        'image_url': [x['image_url'] for x in batch],
        'image': [x['image'] for x in batch],
        'label_6': torch.tensor([x['label_6'] for x in batch], dtype=torch.long),
        'label_2': torch.tensor([x['label_2'] for x in batch], dtype=torch.long),
    }

In [ ]:
train_dataset = FakedditColabDataset(
    train_df,
    image_mode=IMAGE_MODE,
    image_cache_dir=IMAGE_CACHE_DIR,
    download_missing_to_drive=DOWNLOAD_MISSING_TO_DRIVE,
)

val_dataset = FakedditColabDataset(
    val_df,
    image_mode=IMAGE_MODE,
    image_cache_dir=IMAGE_CACHE_DIR,
    download_missing_to_drive=DOWNLOAD_MISSING_TO_DRIVE,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    collate_fn=fakeddit_collate_fn,
)

batch = next(iter(train_loader))
print(batch['text'][0])
print(batch['image_url'][0])
print(batch['label_6'][0].item(), batch['label_2'][0].item())
print(batch['image'][0].size if batch['image'][0] is not None else None)

In [ ]:
import matplotlib.pyplot as plt

sample = train_dataset[0]
print('ID:', sample['id'])
print('Text:', sample['text'])
print('URL:', sample['image_url'])
print('6-way label:', sample['label_6'])
print('2-way label:', sample['label_2'])

if sample['image'] is not None:
    plt.figure(figsize=(6, 6))
    plt.imshow(sample['image'])
    plt.axis('off')
    plt.show()

In [ ]:
# Step 1: remove empty or invalid URLs and empty titles

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["clean_title"] = out["clean_title"].fillna("").astype(str).str.strip()
    out["image_url"] = out["image_url"].fillna("").astype(str).str.strip()

    out = out[out["clean_title"] != ""]
    out = out[out["image_url"] != ""]
    out = out[out["image_url"].str.startswith(("http://", "https://"))]

    return out.reset_index(drop=True)


train_clean = clean_dataframe(train_df)
val_clean = clean_dataframe(val_df)
test_clean = clean_dataframe(test_df)

print(train_clean.shape, val_clean.shape, test_clean.shape)


In [ ]:
# Step 2: create 6-way label mapping for training

label_values = sorted(train_clean["6_way_label"].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_values)}
id2label = {idx: label for label, idx in label2id.items()}

for df in (train_clean, val_clean, test_clean):
    df["label_id"] = df["6_way_label"].map(label2id)
    df["binary_label"] = df["2_way_label"].astype(int)

print(label2id)
train_clean[["6_way_label", "label_id", "binary_label"]].head()


In [ ]:
# Step 3: do a small URL health check on a sample
# This is better than checking all 564k rows right now.

def url_health_check(df: pd.DataFrame, sample_size: int = 100, timeout: int = 8):
    sample_df = df.sample(min(sample_size, len(df)), random_state=42)
    ok = 0
    bad = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        try:
            response = requests.get(
                row["image_url"],
                timeout=timeout,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            Image.open(BytesIO(response.content)).convert("RGB")
            ok += 1
        except Exception:
            bad.append(row["submission_id"])

    print(f"working: {ok}/{len(sample_df)}")
    print(f"failed: {len(bad)}")
    return bad


bad_ids = url_health_check(train_clean, sample_size=100)


In [ ]:
# Step 4: keep only the fields you actually need

keep_cols = [
    "submission_id",
    "clean_title",
    "image_url",
    "2_way_label",
    "6_way_label",
    "label_id",
    "binary_label",
    "split",
]

train_ready = train_clean[keep_cols].copy()
val_ready = val_clean[keep_cols].copy()
test_ready = test_clean[keep_cols].copy()

display(train_ready.head())


In [ ]:
# Step 5: optionally save cleaned CSVs to Drive

SAVE_DIR = Path("/content/drive/MyDrive/Fakeddit/cleaned_url_mode")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_ready.to_csv(SAVE_DIR / "train_ready.csv", index=False)
val_ready.to_csv(SAVE_DIR / "validate_ready.csv", index=False)
test_ready.to_csv(SAVE_DIR / "test_ready.csv", index=False)

print("saved to:", SAVE_DIR)


In [ ]:
from pathlib import Path
import time
import os

CACHE_ROOT = Path("/content/drive/MyDrive/Fakeddit/image_cache")
TRAIN_CACHE_DIR = CACHE_ROOT / "train"
VAL_CACHE_DIR = CACHE_ROOT / "validate"
TEST_CACHE_DIR = CACHE_ROOT / "test"

TRAIN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_LOG_DIR = Path("/content/drive/MyDrive/Fakeddit/cache_logs")
CACHE_LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Cache root:", CACHE_ROOT)
print("Train cache:", TRAIN_CACHE_DIR)
print("Val cache:", VAL_CACHE_DIR)
print("Test cache:", TEST_CACHE_DIR)


In [ ]:
def get_cache_image_path(sample_id: str, cache_dir: Path) -> Path:
    return cache_dir / f"{sample_id}.jpg"


def download_and_save_image(image_url: str, save_path: Path, timeout: int = 15):
    try:
        response = requests.get(
            image_url,
            timeout=timeout,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image.save(save_path, format="JPEG", quality=95)
        return True, None

    except Exception as e:
        return False, str(e)


In [ ]:
def cache_images_from_df(
    df: pd.DataFrame,
    cache_dir: Path,
    split_name: str,
    limit: int | None = None,
    start_idx: int = 0,
    timeout: int = 15,
    sleep_seconds: float = 0.0,
    save_every: int = 500,
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    work_df = df.iloc[start_idx:].reset_index(drop=True)
    if limit is not None:
        work_df = work_df.head(limit).copy()

    log_rows = []
    success = 0
    failed = 0
    skipped = 0

    for idx, row in tqdm(work_df.iterrows(), total=len(work_df), desc=f"Caching {split_name}"):
        sample_id = str(row["submission_id"])
        image_url = row["image_url"]
        save_path = get_cache_image_path(sample_id, cache_dir)

        if save_path.exists():
            skipped += 1
            log_rows.append({
                "submission_id": sample_id,
                "split": split_name,
                "image_url": image_url,
                "image_path": str(save_path),
                "status": "skipped_exists",
                "error": None,
            })
            continue

        ok, err = download_and_save_image(image_url, save_path, timeout=timeout)

        if ok:
            success += 1
            log_rows.append({
                "submission_id": sample_id,
                "split": split_name,
                "image_url": image_url,
                "image_path": str(save_path),
                "status": "downloaded",
                "error": None,
            })
        else:
            failed += 1
            log_rows.append({
                "submission_id": sample_id,
                "split": split_name,
                "image_url": image_url,
                "image_path": str(save_path),
                "status": "failed",
                "error": err,
            })

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

        if (idx + 1) % save_every == 0:
            partial_log = pd.DataFrame(log_rows)
            partial_path = CACHE_LOG_DIR / f"{split_name}_cache_log_partial.csv"
            partial_log.to_csv(partial_path, index=False)

    log_df = pd.DataFrame(log_rows)
    final_log_path = CACHE_LOG_DIR / f"{split_name}_cache_log.csv"
    log_df.to_csv(final_log_path, index=False)

    print({
        "split": split_name,
        "processed": len(work_df),
        "downloaded": success,
        "failed": failed,
        "skipped_exists": skipped,
        "log_path": str(final_log_path),
    })

    return log_df


In [ ]:
# Example: cache first 5000 train images
train_cache_log_1 = cache_images_from_df(
    train_ready,
    cache_dir=TRAIN_CACHE_DIR,
    split_name="train",
    limit=5000,
    start_idx=0,
    timeout=15,
    sleep_seconds=0.0,
    save_every=500,
)

train_cache_log_1.head()


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil
import os

LOCAL_CACHE_ROOT = Path("/content/fakeddit_fast_cache")
LOCAL_TRAIN_CACHE = LOCAL_CACHE_ROOT / "train"
LOCAL_VAL_CACHE = LOCAL_CACHE_ROOT / "validate"
LOCAL_TEST_CACHE = LOCAL_CACHE_ROOT / "test"

LOCAL_TRAIN_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_VAL_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TEST_CACHE.mkdir(parents=True, exist_ok=True)

DRIVE_CACHE_ROOT = Path("/content/drive/MyDrive/Fakeddit/image_cache")
DRIVE_TRAIN_CACHE = DRIVE_CACHE_ROOT / "train"
DRIVE_VAL_CACHE = DRIVE_CACHE_ROOT / "validate"
DRIVE_TEST_CACHE = DRIVE_CACHE_ROOT / "test"

DRIVE_TRAIN_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_VAL_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_TEST_CACHE.mkdir(parents=True, exist_ok=True)

FAST_LOG_DIR = Path("/content/drive/MyDrive/Fakeddit/cache_logs_fast")
FAST_LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 16
TIMEOUT = 12
SYNC_TO_DRIVE = True

print("Local cache root:", LOCAL_CACHE_ROOT)
print("Drive cache root:", DRIVE_CACHE_ROOT)


In [ ]:
def fast_download_one(sample_id: str, image_url: str, cache_dir: Path, timeout: int = 12):
    save_path = cache_dir / f"{sample_id}.jpg"

    if save_path.exists():
        return {
            "submission_id": sample_id,
            "image_url": image_url,
            "image_path": str(save_path),
            "status": "skipped_exists",
            "error": None,
        }

    try:
        response = requests.get(
            image_url,
            timeout=timeout,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        response.raise_for_status()

        with open(save_path, "wb") as f:
            f.write(response.content)

        return {
            "submission_id": sample_id,
            "image_url": image_url,
            "image_path": str(save_path),
            "status": "downloaded",
            "error": None,
        }

    except Exception as e:
        if save_path.exists():
            try:
                save_path.unlink()
            except Exception:
                pass

        return {
            "submission_id": sample_id,
            "image_url": image_url,
            "image_path": str(save_path),
            "status": "failed",
            "error": str(e),
        }


In [ ]:
def cache_images_parallel(
    df: pd.DataFrame,
    cache_dir: Path,
    split_name: str,
    start_idx: int = 0,
    limit: int | None = None,
    max_workers: int = 16,
    timeout: int = 12,
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    work_df = df.iloc[start_idx:].reset_index(drop=True)
    if limit is not None:
        work_df = work_df.head(limit).copy()

    rows = work_df[["submission_id", "image_url"]].copy()
    rows["submission_id"] = rows["submission_id"].astype(str)

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(
                fast_download_one,
                row["submission_id"],
                row["image_url"],
                cache_dir,
                timeout,
            )
            for _, row in rows.iterrows()
        ]

        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Caching {split_name}"):
            results.append(future.result())

    log_df = pd.DataFrame(results)
    log_path = FAST_LOG_DIR / f"{split_name}_cache_log.csv"
    log_df.to_csv(log_path, index=False)

    print(log_df["status"].value_counts(dropna=False))
    print("Log saved to:", log_path)

    return log_df


In [ ]:
def sync_local_cache_to_drive(local_dir: Path, drive_dir: Path):
    drive_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    skipped = 0

    for file_path in tqdm(list(local_dir.glob("*")), desc=f"Sync {local_dir.name} -> Drive"):
        target = drive_dir / file_path.name
        if target.exists():
            skipped += 1
            continue

        shutil.copy2(file_path, target)
        copied += 1

    print({"copied": copied, "skipped": skipped, "target": str(drive_dir)})


In [ ]:
train_cache_log_2 = cache_images_parallel(
    train_ready,
    cache_dir=LOCAL_TRAIN_CACHE,
    split_name="train_part_05000_10000",
    start_idx=10000,
    limit=50000,
    max_workers=MAX_WORKERS,
    timeout=TIMEOUT,
)

if SYNC_TO_DRIVE:
    sync_local_cache_to_drive(LOCAL_TRAIN_CACHE, DRIVE_TRAIN_CACHE)


In [ ]:
def count_cached_images(cache_dir: Path):
    files = list(cache_dir.glob("*.jpg"))
    print("cache_dir:", cache_dir)
    print("cached jpg files:", len(files))
    return len(files)

train_cached_count = count_cached_images(DRIVE_TRAIN_CACHE)


In [ ]:
def check_missing_range(df: pd.DataFrame, cache_dir: Path, start_idx: int, limit: int):
    part = df.iloc[start_idx:start_idx + limit].copy()
    part["expected_path"] = part["submission_id"].astype(str).apply(lambda x: cache_dir / f"{x}.jpg")
    part["exists"] = part["expected_path"].apply(lambda p: p.exists())

    print("range:", start_idx, "to", start_idx + limit - 1)
    print(part["exists"].value_counts(dropna=False))

    missing = part[~part["exists"]].reset_index(drop=True)
    return missing

missing_5000_10000 = check_missing_range(train_ready, DRIVE_TRAIN_CACHE, start_idx=5000, limit=5000)
missing_10000_60000 = check_missing_range(train_ready, DRIVE_TRAIN_CACHE, start_idx=10000, limit=50000)


In [ ]:
train_cache_log_60000_110000 = cache_images_parallel(
    train_ready,
    cache_dir=LOCAL_TRAIN_CACHE,
    split_name="train_part_60000_110000",
    start_idx=60000,
    limit=50000,
    max_workers=MAX_WORKERS,
    timeout=TIMEOUT,
)

if SYNC_TO_DRIVE:
    sync_local_cache_to_drive(LOCAL_TRAIN_CACHE, DRIVE_TRAIN_CACHE)


In [ ]:
train_cached_count = count_cached_images(DRIVE_TRAIN_CACHE)


In [ ]:
def attach_cached_image_paths(df: pd.DataFrame, cache_dir: Path) -> pd.DataFrame:
    out = df.copy()
    out["image_path"] = out["submission_id"].astype(str).apply(
        lambda x: str(cache_dir / f"{x}.jpg")
    )
    out["image_exists"] = out["image_path"].apply(lambda p: Path(p).exists())
    return out


train_cached = attach_cached_image_paths(train_ready, DRIVE_TRAIN_CACHE)
train_local = train_cached[train_cached["image_exists"]].reset_index(drop=True)

print("train_local:", train_local.shape)
print(train_local["6_way_label"].value_counts().sort_index())


In [ ]:
TARGET_SIZE = 100_000

class_counts = train_local["6_way_label"].value_counts().sort_index()
class_ratios = class_counts / class_counts.sum()
target_counts = (class_ratios * TARGET_SIZE).round().astype(int)

# Fix rounding so total becomes exactly TARGET_SIZE
diff = TARGET_SIZE - target_counts.sum()
if diff != 0:
    largest_class = target_counts.idxmax()
    target_counts.loc[largest_class] += diff

print("Available cached class counts:")
print(class_counts)

print("\nTarget 100k class counts:")
print(target_counts)
print("\nTarget total:", target_counts.sum())

train_100k_parts = []

for label, n in target_counts.items():
    label_df = train_local[train_local["6_way_label"] == label]
    n = min(n, len(label_df))
    sampled = label_df.sample(n=n, random_state=42)
    train_100k_parts.append(sampled)

train_100k = (
    pd.concat(train_100k_parts)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("train_100k:", train_100k.shape)
print(train_100k["6_way_label"].value_counts().sort_index())


In [ ]:
SUBSET_DIR = Path("/content/drive/MyDrive/Fakeddit/subsets")
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

train_100k.to_csv(SUBSET_DIR / "train_100k_cached.csv", index=False)

print("Saved:", SUBSET_DIR / "train_100k_cached.csv")


In [ ]:
val_cache_log = cache_images_parallel(
    val_ready,
    cache_dir=LOCAL_VAL_CACHE,
    split_name="validate_full",
    start_idx=0,
    limit=None,
    max_workers=MAX_WORKERS,
    timeout=TIMEOUT,
)

if SYNC_TO_DRIVE:
    sync_local_cache_to_drive(LOCAL_VAL_CACHE, DRIVE_VAL_CACHE)


In [ ]:
val_cached = attach_cached_image_paths(val_ready, DRIVE_VAL_CACHE)
val_local = val_cached[val_cached["image_exists"]].reset_index(drop=True)

print("val_local:", val_local.shape)
print(val_local["6_way_label"].value_counts().sort_index())

val_local.to_csv(SUBSET_DIR / "validate_cached.csv", index=False)


data packaging


In [ ]:
from pathlib import Path

PACKAGE_DIR = Path("/content/drive/MyDrive/Fakeddit/packages")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
!tar -cf /content/drive/MyDrive/Fakeddit/packages/train_images.tar -C /content/drive/MyDrive/Fakeddit/image_cache/train .
!tar -cf /content/drive/MyDrive/Fakeddit/packages/validate_images.tar -C /content/drive/MyDrive/Fakeddit/image_cache/validate .


In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/Fakeddit")
SUBSET_DIR = DRIVE_ROOT / "subsets"

val_full = pd.read_csv(SUBSET_DIR / "validate_cached.csv")

TARGET_VAL_SIZE = 10_000

counts = val_full["6_way_label"].value_counts().sort_index()
ratios = counts / counts.sum()
target_counts = (ratios * TARGET_VAL_SIZE).round().astype(int)

diff = TARGET_VAL_SIZE - target_counts.sum()
if diff != 0:
    target_counts.loc[target_counts.idxmax()] += diff

parts = []

for label, n in target_counts.items():
    label_df = val_full[val_full["6_way_label"] == label]
    n = min(n, len(label_df))
    parts.append(label_df.sample(n=n, random_state=42))

val_10k = (
    pd.concat(parts)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

val_10k.to_csv(SUBSET_DIR / "validate_10k_cached.csv", index=False)

print(val_10k.shape)
print(val_10k["6_way_label"].value_counts().sort_index())


In [ ]:
TRAIN_CSV = SUBSET_DIR / "train_100k_cached.csv"
VAL_CSV = SUBSET_DIR / "validate_10k_cached.csv"

print(TRAIN_CSV.exists())
print(VAL_CSV.exists())


In [ ]:
from pathlib import Path
import shutil
from tqdm.auto import tqdm

DRIVE_TRAIN_IMG_DIR = DRIVE_ROOT / "image_cache/train"
DRIVE_VAL_IMG_DIR = DRIVE_ROOT / "image_cache/validate"

LOCAL_PACKAGE_ROOT = Path("/content/fakeddit_package")
LOCAL_TRAIN_IMG_DIR = LOCAL_PACKAGE_ROOT / "images/train"
LOCAL_VAL_IMG_DIR = LOCAL_PACKAGE_ROOT / "images/validate"
LOCAL_SUBSET_DIR = LOCAL_PACKAGE_ROOT / "subsets"

LOCAL_TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_VAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SUBSET_DIR.mkdir(parents=True, exist_ok=True)

print(DRIVE_TRAIN_IMG_DIR.exists())
print(DRIVE_VAL_IMG_DIR.exists())


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import shutil
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
def copy_one_image(sid, source_dir, target_dir):
    sid = str(sid)
    src = source_dir / f"{sid}.jpg"
    dst = target_dir / f"{sid}.jpg"

    if dst.exists():
        return "skipped"

    if not src.exists():
        return "missing"

    try:
        shutil.copy2(src, dst)
        return "copied"
    except Exception:
        return "failed"


In [ ]:
def copy_images_from_csv_parallel(
    csv_path,
    source_dir,
    target_dir,
    max_workers=32,
    limit=None,
):
    df = pd.read_csv(csv_path)

    if limit is not None:
        df = df.head(limit).copy()

    ids = df["submission_id"].astype(str).tolist()
    target_dir.mkdir(parents=True, exist_ok=True)

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(copy_one_image, sid, source_dir, target_dir)
            for sid in ids
        ]

        for future in tqdm(as_completed(futures), total=len(futures)):
            results.append(future.result())

    result_counts = pd.Series(results).value_counts()
    print(result_counts)
    return result_counts


In [ ]:
copy_images_from_csv_parallel(
    TRAIN_CSV,
    DRIVE_TRAIN_IMG_DIR,
    LOCAL_TRAIN_IMG_DIR,
    max_workers=32,
)


In [ ]:
copy_images_from_csv_parallel(
    VAL_CSV,
    DRIVE_VAL_IMG_DIR,
    LOCAL_VAL_IMG_DIR,
    max_workers=32,
)


In [ ]:
print("local train images:", len(list(LOCAL_TRAIN_IMG_DIR.glob("*.jpg"))))
print("local val images:", len(list(LOCAL_VAL_IMG_DIR.glob("*.jpg"))))


In [ ]:
!du -sh /content/fakeddit_package/images/train
!du -sh /content/fakeddit_package/images/validate


In [ ]:
!tar -cf /content/fakeddit_train_100k_images.tar -C /content/fakeddit_package/images/train .
!tar -cf /content/fakeddit_validate_10k_images.tar -C /content/fakeddit_package/images/validate .
!tar -cf /content/fakeddit_subsets.tar -C /content/fakeddit_package/subsets .


In [ ]:
!ls -lh /content/fakeddit_train_100k_images.tar
!ls -lh /content/fakeddit_validate_10k_images.tar
!ls -lh /content/fakeddit_subsets.tar


In [ ]:
!mkdir -p /content/drive/MyDrive/Fakeddit/packages

!cp /content/fakeddit_validate_10k_images.tar /content/drive/MyDrive/Fakeddit/packages/
!cp /content/fakeddit_subsets.tar /content/drive/MyDrive/Fakeddit/packages/
!cp /content/fakeddit_train_100k_images.tar /content/drive/MyDrive/Fakeddit/packages/


In [ ]:
!ls -lh /content/drive/MyDrive/Fakeddit/packages


In [ ]:
from pathlib import Path
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil
import requests
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

DRIVE_ROOT = Path("/content/drive/MyDrive/Fakeddit")
CACHE_ROOT = DRIVE_ROOT / "image_cache"
DRIVE_TEST_CACHE = CACHE_ROOT / "test"
DRIVE_TEST_CACHE.mkdir(parents=True, exist_ok=True)

LOCAL_CACHE_ROOT = Path("/content/fakeddit_fast_cache")
LOCAL_TEST_CACHE = LOCAL_CACHE_ROOT / "test"
LOCAL_TEST_CACHE.mkdir(parents=True, exist_ok=True)

SUBSET_DIR = DRIVE_ROOT / "subsets"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

# Reload test_ready if Colab runtime forgot it
if "test_ready" not in globals():
    ready_path = DRIVE_ROOT / "cleaned_url_mode/test_ready.csv"
    assert ready_path.exists(), f"Missing {ready_path}. Run your cleaning cells first."
    test_ready = pd.read_csv(ready_path)
    print("Loaded test_ready:", test_ready.shape)

def fast_download_one(sample_id, image_url, cache_dir, timeout=12):
    save_path = cache_dir / f"{sample_id}.jpg"

    if save_path.exists() and save_path.stat().st_size > 1000:
        return {
            "submission_id": sample_id,
            "image_url": image_url,
            "image_path": str(save_path),
            "status": "skipped_exists",
            "error": None,
        }

    try:
        response = requests.get(
            image_url,
            timeout=timeout,
            headers={"User-Agent": "Mozilla/5.0"},
        )

        if response.status_code != 200 or len(response.content) < 1000:
            return {
                "submission_id": sample_id,
                "image_url": image_url,
                "image_path": str(save_path),
                "status": "failed",
                "error": f"bad_response_{response.status_code}",
            }

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image.save(save_path, format="JPEG", quality=95)

        return {
            "submission_id": sample_id,
            "image_url": image_url,
            "image_path": str(save_path),
            "status": "downloaded",
            "error": None,
        }

    except Exception as e:
        return {
            "submission_id": sample_id,
            "image_url": image_url,
            "image_path": str(save_path),
            "status": "failed",
            "error": str(e)[:200],
        }

def cache_images_parallel(
    df,
    cache_dir,
    split_name,
    start_idx=0,
    limit=None,
    max_workers=32,
    timeout=12,
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    work_df = df.iloc[start_idx:].reset_index(drop=True)
    if limit is not None:
        work_df = work_df.head(limit).copy()

    print(f"{split_name}: caching {len(work_df)} images")
    print("cache_dir:", cache_dir)

    rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for _, row in work_df.iterrows():
            futures.append(
                executor.submit(
                    fast_download_one,
                    str(row["submission_id"]),
                    str(row["image_url"]),
                    cache_dir,
                    timeout,
                )
            )

        for future in tqdm(as_completed(futures), total=len(futures)):
            rows.append(future.result())

    log_df = pd.DataFrame(rows)
    print(log_df["status"].value_counts(dropna=False))

    log_path = DRIVE_ROOT / "cache_logs_fast" / f"{split_name}_cache_log.csv"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_df.to_csv(log_path, index=False)
    print("saved log:", log_path)

    return log_df

def sync_local_cache_to_drive(local_dir, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)

    files = list(Path(local_dir).glob("*.jpg"))
    copied = 0
    skipped = 0

    for src in tqdm(files, desc="sync to drive"):
        dst = Path(drive_dir) / src.name

        if dst.exists() and dst.stat().st_size > 1000:
            skipped += 1
            continue

        shutil.copy2(src, dst)
        copied += 1

    print({"copied": copied, "skipped": skipped})

def count_cached_images(cache_dir):
    files = list(Path(cache_dir).glob("*.jpg"))
    print("cache_dir:", cache_dir)
    print("cached jpg files:", len(files))
    return len(files)

def attach_cached_image_paths(df, cache_dir):
    out = df.copy()
    out["submission_id"] = out["submission_id"].astype(str)
    out["image_path"] = out["submission_id"].apply(lambda x: str(Path(cache_dir) / f"{x}.jpg"))
    out["image_exists"] = out["image_path"].apply(lambda p: Path(p).exists())
    return out

print("Test caching functions ready.")


In [ ]:
MAX_WORKERS = 64
TIMEOUT = 12
SYNC_TO_DRIVE = True

test_cache_log = cache_images_parallel(
    test_ready,
    cache_dir=LOCAL_TEST_CACHE,
    split_name="test_full",
    start_idx=0,
    limit=None,
    max_workers=MAX_WORKERS,
    timeout=TIMEOUT,
)

if SYNC_TO_DRIVE:
    sync_local_cache_to_drive(LOCAL_TEST_CACHE, DRIVE_TEST_CACHE)

test_cached_count = count_cached_images(DRIVE_TEST_CACHE)


In [ ]:
test_cached = attach_cached_image_paths(test_ready, DRIVE_TEST_CACHE)
test_local = test_cached[test_cached["image_exists"]].reset_index(drop=True)

print("test_local:", test_local.shape)
print(test_local["6_way_label"].value_counts().sort_index())

TARGET_TEST_SIZE = 10_000

assert len(test_local) >= TARGET_TEST_SIZE, "Not enough cached test images. Cache more test images first."

class_counts = test_local["6_way_label"].value_counts().sort_index()
class_ratios = class_counts / class_counts.sum()
target_counts = (class_ratios * TARGET_TEST_SIZE).round().astype(int)

diff = TARGET_TEST_SIZE - target_counts.sum()
if diff != 0:
    target_counts.loc[target_counts.idxmax()] += diff

parts = []
for label, n in target_counts.items():
    label_df = test_local[test_local["6_way_label"] == label]
    parts.append(label_df.sample(n=min(n, len(label_df)), random_state=42))

test_10k = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

if len(test_10k) < TARGET_TEST_SIZE:
    remaining = test_local.drop(test_10k.index, errors="ignore")
    extra = remaining.sample(n=TARGET_TEST_SIZE - len(test_10k), random_state=43)
    test_10k = pd.concat([test_10k, extra]).sample(frac=1, random_state=44).reset_index(drop=True)

test_10k = test_10k.head(TARGET_TEST_SIZE)

SUBSET_DIR = Path("/content/drive/MyDrive/Fakeddit/subsets")
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

test_10k.to_csv(SUBSET_DIR / "test_10k_cached.csv", index=False)

print("saved:", SUBSET_DIR / "test_10k_cached.csv")
print("test_10k:", test_10k.shape)
print(test_10k["6_way_label"].value_counts().sort_index())


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Fakeddit")
SUBSET_DIR = DRIVE_ROOT / "subsets"

TEST_CSV = SUBSET_DIR / "test_10k_cached.csv"
DRIVE_TEST_IMG_DIR = DRIVE_ROOT / "image_cache/test"

LOCAL_PACKAGE_ROOT = Path("/content/fakeddit_package")
LOCAL_TEST_IMG_DIR = LOCAL_PACKAGE_ROOT / "images/test"
LOCAL_TEST_IMG_DIR.mkdir(parents=True, exist_ok=True)

print(TEST_CSV.exists(), DRIVE_TEST_IMG_DIR.exists(), LOCAL_TEST_IMG_DIR)


In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil
import pandas as pd
from tqdm.auto import tqdm

def copy_one_image(sample_id, source_dir, target_dir):
    src = Path(source_dir) / f"{sample_id}.jpg"
    dst = Path(target_dir) / f"{sample_id}.jpg"

    if dst.exists() and dst.stat().st_size > 1000:
        return "exists"

    if not src.exists() or src.stat().st_size <= 1000:
        return "missing"

    try:
        shutil.copy2(src, dst)
        return "copied"
    except Exception:
        return "failed"

def copy_images_from_csv_parallel(
    csv_path,
    source_dir,
    target_dir,
    max_workers=32,
    limit=None,
):
    df = pd.read_csv(csv_path)
    if limit is not None:
        df = df.head(limit).copy()

    ids = df["submission_id"].astype(str).tolist()
    Path(target_dir).mkdir(parents=True, exist_ok=True)

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [
            ex.submit(copy_one_image, sid, source_dir, target_dir)
            for sid in ids
        ]

        for f in tqdm(as_completed(futures), total=len(futures)):
            results.append(f.result())

    counts = pd.Series(results).value_counts()
    print(counts)

    return counts

print("copy_images_from_csv_parallel ready")


In [ ]:
copy_images_from_csv_parallel(
    TEST_CSV,
    DRIVE_TEST_IMG_DIR,
    LOCAL_TEST_IMG_DIR,
    max_workers=32,
)


In [ ]:
!find /content/fakeddit_package/images/test -name "*.jpg" | wc -l



In [ ]:
!tar -cf /content/fakeddit_test_10k_images.tar -C /content/fakeddit_package/images/test .
!ls -lh /content/fakeddit_test_10k_images.tar


In [ ]:
!mkdir -p /content/drive/MyDrive/Fakeddit/packages
!cp /content/fakeddit_test_10k_images.tar /content/drive/MyDrive/Fakeddit/packages/
!ls -lh /content/drive/MyDrive/Fakeddit/packages/fakeddit_test_10k_images.tar
!ls -lh /content/drive/MyDrive/Fakeddit/subsets/test_10k_cached.csv


In [ ]:
!tar -tf /content/fakeddit_test_10k_images.tar | wc -l


In [ ]:
!cp /content/fakeddit_test_10k_images.tar /content/drive/MyDrive/Fakeddit/packages/
!ls -lh /content/drive/MyDrive/Fakeddit/packages/fakeddit_test_10k_images.tar
!ls -lh /content/drive/MyDrive/Fakeddit/subsets/test_10k_cached.csv


In [ ]:
!ls -lh /content/drive/MyDrive/Fakeddit/packages/
!ls -lh /content/drive/MyDrive/Fakeddit/subsets/


In [ ]:
!ls -lh /content/drive/MyDrive/Fakeddit/packages/fakeddit_test_10k_images.tar
!ls -lh /content/drive/MyDrive/Fakeddit/subsets/test_10k_cached.csv


In [ ]:
!ls -lh /content/fakeddit_test_10k_images.tar
!ls -lh /content/fakeddit_package/subsets/test_10k_cached.csv
!find /content/fakeddit_package/images/test -maxdepth 1 -name "*.jpg" | wc -l



In [ ]:

!ls -ld /content/drive/MyDrive/Fakeddit
!ls -ld /content/drive/MyDrive/Fakeddit/image_cache/test
!find /content/drive/MyDrive/Fakeddit/image_cache/test -maxdepth 1 -name "*.jpg" | wc -l

In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/Fakeddit")
TEST_CSV = DRIVE_ROOT / "subsets/test_10k_cached.csv"
DRIVE_TEST_IMG_DIR = DRIVE_ROOT / "image_cache/test"

# If CSV is missing, recreate it from test_ready + cached images
if not TEST_CSV.exists():
    if "test_ready" not in globals():
        test_ready = pd.read_csv(DRIVE_ROOT / "cleaned_url_mode/test_ready.csv")

    def attach_cached_image_paths(df, cache_dir):
        out = df.copy()
        out["submission_id"] = out["submission_id"].astype(str)
        out["image_path"] = out["submission_id"].apply(lambda x: str(Path(cache_dir) / f"{x}.jpg"))
        out["image_exists"] = out["image_path"].apply(lambda p: Path(p).exists())
        return out

    test_cached = attach_cached_image_paths(test_ready, DRIVE_TEST_IMG_DIR)
    test_local = test_cached[test_cached["image_exists"]].reset_index(drop=True)

    TARGET_TEST_SIZE = 10_000
    class_counts = test_local["6_way_label"].value_counts().sort_index()
    ratios = class_counts / class_counts.sum()
    target_counts = (ratios * TARGET_TEST_SIZE).round().astype(int)
    diff = TARGET_TEST_SIZE - target_counts.sum()
    if diff != 0:
        target_counts.loc[target_counts.idxmax()] += diff

    parts = []
    for label, n in target_counts.items():
        parts.append(test_local[test_local["6_way_label"] == label].sample(n=int(n), random_state=42))

    test_10k = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
    TEST_CSV.parent.mkdir(parents=True, exist_ok=True)
    test_10k.to_csv(TEST_CSV, index=False)

test_10k = pd.read_csv(TEST_CSV)
print(test_10k.shape)
print(test_10k["6_way_label"].value_counts().sort_index())


In [ ]:
from pathlib import Path

LIST_PATH = Path("/content/test_10k_tar_list.txt")

ids = test_10k["submission_id"].astype(str).tolist()

missing = []
with open(LIST_PATH, "w") as f:
    for sid in ids:
        img_path = DRIVE_TEST_IMG_DIR / f"{sid}.jpg"
        if img_path.exists():
            f.write(f"{sid}.jpg\n")
        else:
            missing.append(sid)

print("total ids:", len(ids))
print("missing images:", len(missing))
print("tar list:", LIST_PATH)

assert len(missing) == 0, "Some selected test images are missing."


In [ ]:
!find /content/fakeddit_package/images/test -maxdepth 1 -name "*.jpg" | wc -l



In [ ]:
!rm -f /content/fakeddit_test_10k_images.tar


In [ ]:
!rm -f /content/fakeddit_test_10k_images.tar
!rm -rf /content/fakeddit_package/images/test
!mkdir -p /content/fakeddit_package/images/test


In [ ]:
!rm -f /content/fakeddit_test_10k_images.tar
!rm -rf /content/fakeddit_package/images/test
!mkdir -p /content/fakeddit_package/images/test


In [ ]:
from pathlib import Path
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd

LOCAL_TEST_IMG_DIR = Path("/content/fakeddit_package/images/test")
LOCAL_TEST_IMG_DIR.mkdir(parents=True, exist_ok=True)

ids = test_10k["submission_id"].astype(str).tolist()

def copy_one_local(sid):
    src = DRIVE_TEST_IMG_DIR / f"{sid}.jpg"
    dst = LOCAL_TEST_IMG_DIR / f"{sid}.jpg"

    if dst.exists() and dst.stat().st_size > 1000:
        return "exists"

    if not src.exists() or src.stat().st_size <= 1000:
        return "missing"

    try:
        shutil.copy2(src, dst)
        return "copied"
    except Exception:
        return "failed"

results = []

with ThreadPoolExecutor(max_workers=32) as ex:
    futures = [ex.submit(copy_one_local, sid) for sid in ids]
    for f in tqdm(as_completed(futures), total=len(futures)):
        results.append(f.result())

counts = pd.Series(results).value_counts()
print(counts)


In [ ]:
!find /content/fakeddit_package/images/test -maxdepth 1 -name "*.jpg" | wc -l


In [ ]:
!tar -cf /content/fakeddit_test_10k_images.tar -C /content/fakeddit_package/images/test .
!ls -lh /content/fakeddit_test_10k_images.tar
!tar -tf /content/fakeddit_test_10k_images.tar | grep -E "\.jpg$" | wc -l


In [ ]:
test_10k.to_csv("/content/test_10k_cached.csv", index=False)
!ls -lh /content/test_10k_cached.csv


In [ ]:
!mkdir -p /content/drive/MyDrive/Fakeddit/packages
!mkdir -p /content/drive/MyDrive/Fakeddit/subsets

!cp /content/fakeddit_test_10k_images.tar /content/drive/MyDrive/Fakeddit/packages/fakeddit_test_10k_images.tar
!cp /content/test_10k_cached.csv /content/drive/MyDrive/Fakeddit/subsets/test_10k_cached.csv

!ls -lh /content/drive/MyDrive/Fakeddit/packages/fakeddit_test_10k_images.tar
!ls -lh /content/drive/MyDrive/Fakeddit/subsets/test_10k_cached.csv
